## 2026 EY AI & Data Challenge - NASADEM Data Extraction Notebook

This notebook demonstrates how to access the NASADEM dataset. NASADEM is a global Digital Elevation Model (DEM) derived from the original Shuttle Radar Topography Mission (SRTM) data, reprocessed with improved algorithms and auxiliary datasets. It provides elevation data at ~30-meter (1 arc-second) spatial resolution. The data is available as Cloud Optimized GeoTIFFs (COGs) via the Microsoft Planetary Computer.

For more information, visit: [nasadem - overview](https://planetarycomputer.microsoft.com/dataset/nasadem#overview)

## Load In Dependencies
The following code installs the required Python libraries (found in the requirements.txt file) in the Snowflake environment to allow successful execution of the remaining notebook code. After running this code for the first time, it is required to “restart” the kernal so the Python libraries are available in the environment. This is done by selecting the “Connected” menu above the notebook (next to “Run all”) and selecting the “restart kernal” link. Subsequent runs of the notebook do not require this “restart” process.

In [ ]:
!pip install uv
!uv pip install  -r requirements.txt
!uv pip install rasterio pysheds

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
import rasterio.transform

if not hasattr(np, 'in1d'):
    np.in1d = np.isin

import pystac_client
import planetary_computer as pc
from pysheds.grid import Grid

from tqdm import tqdm

## Extracting NASADEM Data Using API Calls

The API-based method allows us to efficiently access **NASADEM** elevation data through the [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/), ensuring scalability and reproducibility of the process.

From the raw elevation raster, we derive three topographic features:

- **Elevation** — raw height above sea level from the NASADEM DEM.
- **Slope** — terrain steepness (degrees), computed from the elevation gradient.
- **Flow Accumulation** — the number of upstream cells that drain through each cell, computed using the D8 (steepest-descent) algorithm. This serves as a proxy for upstream contributing area and catchment size.

These topographic features provide important context for water flow, drainage patterns, and water quality characteristics across the study region.

### Loading and Mapping NASADEM Data

This section demonstrates how **NASADEM elevation data** is loaded and topographic features are derived and mapped to sampling locations.

- The **get_nasadem_items** function searches the Microsoft Planetary Computer STAC catalog for NASADEM tiles covering the study region's bounding box.
- The **compute_slope** function computes terrain slope (degrees) from the elevation raster using `np.gradient`, converting pixel spacing from degrees to meters for accurate physical slope values.
- The **compute_flow_accumulation** function uses **pysheds** to compute hydrologically-correct flow accumulation. The DEM is first conditioned by filling pits, filling depressions, and resolving flats — critical preprocessing steps that ensure water routes continuously downhill without getting trapped in artefacts. D8 flow directions and accumulation are then computed on the conditioned surface.
- The **extract_nasadem_features** function orchestrates the full pipeline: for each tile, it reads elevation, computes slope and flow accumulation, and extracts all three values at each sampling location.

Since NASADEM is a static elevation dataset (not time-varying), each sampling location receives the same topographic feature values regardless of its sample date.

In [2]:
def get_nasadem_items(bbox):
    """Search for NASADEM tiles covering the given bounding box [west, south, east, north]."""
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    search = catalog.search(collections=["nasadem"], bbox=bbox)
    items = list(search.items())
    return items

In [3]:
def compute_slope(elev_data, transform, bounds):
    """Compute slope in degrees from elevation raster, converting pixel spacing to meters."""
    lat_center = (bounds.top + bounds.bottom) / 2
    pixel_deg = abs(transform.a)
    cellsize_x = pixel_deg * 111320 * np.cos(np.radians(lat_center))
    cellsize_y = pixel_deg * 110540

    dy, dx = np.gradient(elev_data, cellsize_y, cellsize_x)
    slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
    slope[np.isnan(elev_data)] = np.nan
    return slope


def compute_flow_accumulation(href):
    """Compute hydrologically-conditioned D8 flow accumulation using pysheds.

    Pipeline: fill pits -> fill depressions -> resolve flats -> D8 flow direction -> accumulation.
    This ensures proper hydrologic routing by removing artefacts (pits, depressions, flats) from
    the raw DEM before computing flow directions.
    """
    grid = Grid.from_raster(href)
    dem = grid.read_raster(href)

    pit_filled = grid.fill_pits(dem)
    flooded = grid.fill_depressions(pit_filled)
    inflated = grid.resolve_flats(flooded)

    dirmap = (64, 128, 1, 2, 4, 8, 16, 32)
    fdir = grid.flowdir(inflated, dirmap=dirmap)
    acc = grid.accumulation(fdir, dirmap=dirmap)

    return np.array(acc)

In [4]:
def extract_nasadem_features(df, items):
    """Extract elevation, slope, and flow accumulation for each lat/lon point."""
    n = len(df)
    elevations = np.full(n, np.nan)
    slopes = np.full(n, np.nan)
    flow_accs = np.full(n, np.nan)

    lats = df['Latitude'].values
    lons = df['Longitude'].values

    for item in tqdm(items, desc="Processing NASADEM tiles"):
        href = item.assets['elevation'].href

        with rasterio.open(href) as src:
            bounds = src.bounds
            mask = (
                (lats >= bounds.bottom) & (lats <= bounds.top) &
                (lons >= bounds.left) & (lons <= bounds.right)
            )
            indices = np.where(mask)[0]
            if len(indices) == 0:
                continue

            elev_data = src.read(1).astype(np.float64)
            nodata = src.nodata
            height, width = src.height, src.width
            transform = src.transform
            if nodata is not None:
                elev_data[elev_data == nodata] = np.nan

            slope_data = compute_slope(elev_data, transform, bounds)

        acc_data = compute_flow_accumulation(href)

        for idx in indices:
            if not np.isnan(elevations[idx]):
                continue
            r, c = rasterio.transform.rowcol(transform, lons[idx], lats[idx])
            r, c = int(r), int(c)
            if 0 <= r < height and 0 <= c < width:
                elevations[idx] = elev_data[r, c]
                slopes[idx] = slope_data[r, c]
                flow_accs[idx] = acc_data[r, c]

    return elevations, slopes, flow_accs

### Extracting features for the training dataset

In [5]:
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
display(Water_Quality_df.head(5))

In [6]:
Water_Quality_df.shape

In [ ]:
# Search for NASADEM tiles covering the study area (South Africa)
bbox = [14.97, -35.18, 32.79, -21.72]  # [west, south, east, north]
nasadem_items = get_nasadem_items(bbox)
print(f"Found {len(nasadem_items)} NASADEM tiles in the study area bounding box")

# Pre-filter to only tiles that contain at least one training sample point
lats = Water_Quality_df['Latitude'].values
lons = Water_Quality_df['Longitude'].values
training_items = [
    item for item in nasadem_items
    if np.any(
        (lats >= item.bbox[1]) & (lats <= item.bbox[3]) &
        (lons >= item.bbox[0]) & (lons <= item.bbox[2])
    )
]
print(f"Filtered to {len(training_items)} tiles containing training sample points")

# Split into two batches so each cell stays under 40 minutes
mid = len(training_items) // 2
training_batch_1 = training_items[:mid]
training_batch_2 = training_items[mid:]
print(f"Batch 1: {len(training_batch_1)} tiles | Batch 2: {len(training_batch_2)} tiles")

In [ ]:
# Training extraction — Batch 1
train_elev_1, train_slope_1, train_acc_1 = extract_nasadem_features(Water_Quality_df, training_batch_1)
print(f"Batch 1 complete — {np.sum(~np.isnan(train_elev_1))}/{len(train_elev_1)} points matched so far")

In [ ]:
# Training extraction — Batch 1
train_elev_1, train_slope_1, train_acc_1 = extract_nasadem_features(Water_Quality_df, training_batch_1)
print(f"Batch 1 complete — {np.sum(~np.isnan(train_elev_1))}/{len(train_elev_1)} points matched so far")

In [ ]:
# Training extraction — Batch 2
train_elev_2, train_slope_2, train_acc_2 = extract_nasadem_features(Water_Quality_df, training_batch_2)
print(f"Batch 2 complete — {np.sum(~np.isnan(train_elev_2))}/{len(train_elev_2)} points matched so far")

In [ ]:
# Merge batch results (each point falls in exactly one tile, so take the non-NaN value)
training_elevations = np.where(~np.isnan(train_elev_1), train_elev_1, train_elev_2)
training_slopes = np.where(~np.isnan(train_slope_1), train_slope_1, train_slope_2)
training_flow_accs = np.where(~np.isnan(train_acc_1), train_acc_1, train_acc_2)
print(f"Training feature extraction complete — {np.sum(~np.isnan(training_elevations))}/{len(training_elevations)} points matched")

In [ ]:
# Build the NASADEM training features DataFrame
nasadem_training_df = pd.DataFrame({
    'Latitude': Water_Quality_df['Latitude'],
    'Longitude': Water_Quality_df['Longitude'],
    'Sample Date': Water_Quality_df['Sample Date'],
    'elevation': training_elevations,
    'slope': training_slopes,
    'flow_accumulation': training_flow_accs,
})
print(f"Training shape: {nasadem_training_df.shape}")
display(nasadem_training_df.head())

In [8]:
nasadem_training_df.to_csv('nasadem_features_training_.csv', index=False)
print("Saved nasadem_features_training_.csv")

In [9]:
# Preview File
display(nasadem_training_df.head())

In [ ]:
nasadem_training_df.to_csv("/tmp/nasadem_features_training_.csv", index=False)

In [ ]:
session.sql("""
    PUT file:///tmp/nasadem_features_training_.csv
    'snow://workspace/USER$.PUBLIC."Ian-EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.

### Extracting features for the validation dataset

In [10]:
Validation_df=pd.read_csv('submission_template.csv')
display(Validation_df.head())

In [11]:
Validation_df.shape

In [12]:
# Pre-filter tiles for validation points
val_lats = Validation_df['Latitude'].values
val_lons = Validation_df['Longitude'].values
validation_items = [
    item for item in nasadem_items
    if np.any(
        (val_lats >= item.bbox[1]) & (val_lats <= item.bbox[3]) &
        (val_lons >= item.bbox[0]) & (val_lons <= item.bbox[2])
    )
]
mid_v = len(validation_items) // 2
val_batch_1 = validation_items[:mid_v]
val_batch_2 = validation_items[mid_v:]
print(f"Validation: {len(validation_items)} tiles with points (Batch 1: {len(val_batch_1)}, Batch 2: {len(val_batch_2)})")

In [ ]:
# Validation extraction — Batch 1
val_elev_1, val_slope_1, val_acc_1 = extract_nasadem_features(Validation_df, val_batch_1)
print(f"Validation Batch 1 complete — {np.sum(~np.isnan(val_elev_1))}/{len(val_elev_1)} points matched so far")

In [ ]:
# Validation extraction — Batch 2
val_elev_2, val_slope_2, val_acc_2 = extract_nasadem_features(Validation_df, val_batch_2)
print(f"Validation Batch 2 complete — {np.sum(~np.isnan(val_elev_2))}/{len(val_elev_2)} points matched so far")

In [ ]:
# Merge validation batch results
validation_elevations = np.where(~np.isnan(val_elev_1), val_elev_1, val_elev_2)
validation_slopes = np.where(~np.isnan(val_slope_1), val_slope_1, val_slope_2)
validation_flow_accs = np.where(~np.isnan(val_acc_1), val_acc_1, val_acc_2)
print(f"Validation feature extraction complete — {np.sum(~np.isnan(validation_elevations))}/{len(validation_elevations)} points matched")

In [ ]:
# Build the NASADEM validation features DataFrame
nasadem_validation_df = pd.DataFrame({
    'Latitude': Validation_df['Latitude'],
    'Longitude': Validation_df['Longitude'],
    'Sample Date': Validation_df['Sample Date'],
    'elevation': validation_elevations,
    'slope': validation_slopes,
    'flow_accumulation': validation_flow_accs,
})
print(f"Validation shape: {nasadem_validation_df.shape}")
display(nasadem_validation_df.head())

In [13]:
nasadem_validation_df.to_csv('nasadem_features_validation_.csv', index=False)
print("Saved nasadem_features_validation_.csv")

In [14]:
# Preview File
display(nasadem_validation_df.head())

In [ ]:
nasadem_validation_df.to_csv("/tmp/nasadem_features_validation_.csv", index=False)

In [ ]:
session.sql("""
    PUT file:///tmp/nasadem_features_validation_.csv
    'snow://workspace/USER$.PUBLIC."Ian-EY-AI-and-Data-Challenge"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()

print("File saved! Refresh the browser to see the files in the sidebar")

**Note:** If you're using your own workspace, remember to replace "EY-AI-and-Data-Challenge" with your workspace name in the file path.